# Capstone Project - ML Pipeline

## Data Preprocessing & Modelling

## 1. Imports and Configuration

In [ ]:
# =========================
# Imports and settings
# =========================

import os
import glob
import warnings

import numpy as np
import pandas as pd

# =========================
# Modelling imports
# =========================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load Raw Data

In [3]:
# =========================
# 2. Project paths
# =========================

PROJECT_DIR = os.getcwd()

DATA_DIR = os.path.join(PROJECT_DIR, "Data")
RAW_DATA_PATH = os.path.join(DATA_DIR, "data.csv")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data path:", RAW_DATA_PATH)
print("Processed data directory:", PROCESSED_DATA_DIR)

Project directory: c:\Users\tozeq\Desktop\LDSSA - Capstone Project
Raw data path: c:\Users\tozeq\Desktop\LDSSA - Capstone Project\Data\data.csv
Processed data directory: c:\Users\tozeq\Desktop\LDSSA - Capstone Project\Data\processed


In [4]:
# =========================
# 3. Load raw data
# =========================

df = pd.read_csv(RAW_DATA_PATH, low_memory=False)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1095429, 36)


,call_number,unit_id,incident_number,call_type,call_date,watch_date,received_dttm,entry_dttm,dispatch_dttm,response_dttm,on_scene_dttm,transport_dttm,hospital_dttm,call_final_disposition,available_dttm,address,city,zipcode_of_incident,battalion,station_area,box,original_priority,priority,final_priority,als_unit,call_type_group,number_of_alarms,unit_type,unit_sequence_in_call_dispatch,fire_prevention_district,supervisor_district,neighborhoods_analysis_boundaries,rowid,case_location,data_as_of,data_loaded_at
0,230010392,T03,23000066,Gas Leak (Natural and LP Gases),2023-01-01T00:00:00.000,2022-12-31T00:00:00.000,2023-01-01T01:56:38.000,2023-01-01T01:58:36.000,2023-01-01T01:59:03.000,2023-01-01T02:02:27.000,2023-01-01T02:02:27.000,NaN,NaN,Fire,2023-01-01T02:11:34.000,GEARY ST/JONES ST,San Francisco,"94,102.00",B01,3.00,1462,3,3,3,True,Alarm,1,TRUCK,1.00,1.00,6.00,Tenderloin,230010392-T03,POINT (-122.41316 37.786728),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000
1,230010428,52,23000073,Medical Incident,2023-01-01T00:00:00.000,2022-12-31T00:00:00.000,2023-01-01T02:10:43.000,2023-01-01T02:12:06.000,2023-01-01T02:13:10.000,2023-01-01T02:13:50.000,2023-01-01T02:25:45.000,2023-01-01T02:53:18.000,2023-01-01T03:11:44.000,Code 2 Transport,2023-01-01T03:51:20.000,03RD ST/DONNER AVE,San Francisco,"94,124.00",B10,17.00,6537,3,3,3,True,Potentially Life-Threatening,1,MEDIC,2.00,10.00,10.00,Bayview Hunters Point,230010428-52,POINT (-122.39453 37.724693),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000
2,230011485,B05,23000257,Alarms,2023-01-01T00:00:00.000,2023-01-01T00:00:00.000,2023-01-01T11:33:00.000,2023-01-01T11:34:23.000,2023-01-01T11:34:29.000,2023-01-01T11:35:00.000,2023-01-01T11:37:50.000,NaN,NaN,Fire,2023-01-01T12:01:28.000,GOLDEN GATE AVE/MASONIC AVE,San Francisco,"94,118.00",B05,21.00,4462,3,3,3,False,Alarm,1,CHIEF,3.00,5.00,1.00,Lone Mountain/USF,230011485-B05,POINT (-122.446846 37.777668),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000
3,230011085,86,23000201,Medical Incident,2023-01-01T00:00:00.000,2023-01-01T00:00:00.000,2023-01-01T08:28:08.000,2023-01-01T08:30:11.000,2023-01-01T08:32:00.000,2023-01-01T08:32:03.000,NaN,NaN,NaN,Code 2 Transport,2023-01-01T08:32:19.000,IRVING ST/41ST AVE,San Francisco,"94,122.00",B08,23.00,7627,2,3,3,True,Potentially Life-Threatening,1,MEDIC,4.00,8.00,4.00,Sunset/Parkside,230011085-86,POINT (-122.5007 37.762524),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000
4,230012990,T03,23000486,Medical Incident,2023-01-01T00:00:00.000,2023-01-01T00:00:00.000,2023-01-01T20:30:52.000,2023-01-01T20:31:40.000,2023-01-01T20:32:44.000,NaN,NaN,NaN,NaN,Code 2 Transport,2023-01-01T20:36:19.000,JONES ST/POST ST,San Francisco,"94,109.00",B04,3.00,1543,3,3,3,False,Potentially Life-Threatening,1,TRUCK,3.00,1.00,3.00,Tenderloin,230012990-T03,POINT (-122.41335 37.787663),2024-02-05T03:27:52.000,2024-02-05T10:56:25.000


In [5]:
# =========================
# 4. Initial inspection
# =========================

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(30))

Columns:
['call_number', 'unit_id', 'incident_number', 'call_type', 'call_date', 'watch_date', 'received_dttm', 'entry_dttm', 'dispatch_dttm', 'response_dttm', 'on_scene_dttm', 'transport_dttm', 'hospital_dttm', 'call_final_disposition', 'available_dttm', 'address', 'city', 'zipcode_of_incident', 'battalion', 'station_area', 'box', 'original_priority', 'priority', 'final_priority', 'als_unit', 'call_type_group', 'number_of_alarms', 'unit_type', 'unit_sequence_in_call_dispatch', 'fire_prevention_district', 'supervisor_district', 'neighborhoods_analysis_boundaries', 'rowid', 'case_location', 'data_as_of', 'data_loaded_at']

Data types:


call_number                            int64
unit_id                                  str
incident_number                        int64
call_type                                str
call_date                                str
watch_date                               str
received_dttm                            str
entry_dttm                               str
dispatch_dttm                            str
response_dttm                            str
on_scene_dttm                            str
transport_dttm                           str
hospital_dttm                            str
call_final_disposition                   str
available_dttm                           str
address                                  str
city                                     str
zipcode_of_incident                  float64
battalion                                str
station_area                         float64
box                                      str
original_priority                        str
priority  


Missing values:


hospital_dttm                        839024
transport_dttm                       834750
on_scene_dttm                        240401
response_dttm                         28687
call_type_group                       20413
fire_prevention_district               6578
original_priority                      6480
city                                   1323
zipcode_of_incident                     836
available_dttm                          216
neighborhoods_analysis_boundaries       210
supervisor_district                     197
address                                 110
station_area                             52
box                                      50
case_location                            42
unit_sequence_in_call_dispatch            1
incident_number                           0
unit_id                                   0
call_number                               0
call_type                                 0
battalion                                 0
watch_date                      

## 3. Standardize Column Names

In [6]:
# =========================
# 3. Standardize / validate column names
# =========================

# Make sure column names are lowercase and use underscores
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Number of columns:", len(df.columns))
print(df.columns.tolist())

Number of columns: 36
['call_number', 'unit_id', 'incident_number', 'call_type', 'call_date', 'watch_date', 'received_dttm', 'entry_dttm', 'dispatch_dttm', 'response_dttm', 'on_scene_dttm', 'transport_dttm', 'hospital_dttm', 'call_final_disposition', 'available_dttm', 'address', 'city', 'zipcode_of_incident', 'battalion', 'station_area', 'box', 'original_priority', 'priority', 'final_priority', 'als_unit', 'call_type_group', 'number_of_alarms', 'unit_type', 'unit_sequence_in_call_dispatch', 'fire_prevention_district', 'supervisor_district', 'neighborhoods_analysis_boundaries', 'rowid', 'case_location', 'data_as_of', 'data_loaded_at']


## 4. Parse Datetime Columns

In [7]:
# =========================
# 4. Parse datetime columns
# =========================

datetime_cols = [
    "call_date",
    "watch_date",
    "received_dttm",
    "entry_dttm",
    "dispatch_dttm",
    "response_dttm",
    "on_scene_dttm",
    "transport_dttm",
    "hospital_dttm",
    "available_dttm",
    "data_as_of",
    "data_loaded_at",
]

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df[datetime_cols].dtypes

call_date         datetime64[us]
watch_date        datetime64[us]
received_dttm     datetime64[us]
entry_dttm        datetime64[us]
dispatch_dttm     datetime64[us]
response_dttm     datetime64[us]
on_scene_dttm     datetime64[us]
transport_dttm    datetime64[us]
hospital_dttm     datetime64[us]
available_dttm    datetime64[us]
data_as_of        datetime64[us]
data_loaded_at    datetime64[us]
dtype: object

## 5. Create Target: response_time_seconds

In [8]:
# =========================
# 5. Create target: response_time_seconds
# =========================

df["response_time_seconds"] = (
    df["on_scene_dttm"] - df["received_dttm"]
).dt.total_seconds()

df[["received_dttm", "on_scene_dttm", "response_time_seconds"]].head(10)

,received_dttm,on_scene_dttm,response_time_seconds
0,2023-01-01 01:56:38,2023-01-01 02:02:27,349.00
1,2023-01-01 02:10:43,2023-01-01 02:25:45,902.00
2,2023-01-01 11:33:00,2023-01-01 11:37:50,290.00
3,2023-01-01 08:28:08,NaT,NaN
4,2023-01-01 20:30:52,NaT,NaN
5,2023-01-01 00:35:26,NaT,NaN
6,2023-01-01 20:09:14,2023-01-01 20:16:29,435.00
7,2023-01-01 16:17:13,2023-01-01 16:28:55,702.00
8,2023-01-01 11:48:39,2023-01-01 12:13:37,"1,498.00"
9,2023-01-01 00:44:56,NaT,NaN


## 6. Document Removed Records

In [9]:
# =========================
# 6. Document removed records
# =========================

display(df["response_time_seconds"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))

df["exclusion_reason"] = "kept"

df.loc[df["on_scene_dttm"].isna(), "exclusion_reason"] = "missing_on_scene_dttm"
df.loc[df["response_time_seconds"] < 0, "exclusion_reason"] = "negative_response_time"

EXTREME_RESPONSE_TIME_SECONDS = 60 * 60

df.loc[
    df["response_time_seconds"] > EXTREME_RESPONSE_TIME_SECONDS,
    "exclusion_reason"
] = "extreme_response_time_over_60_min"

exclusion_summary = (
    df["exclusion_reason"]
    .value_counts()
    .rename_axis("exclusion_reason")
    .reset_index(name="records")
)

exclusion_summary["percentage"] = (
    exclusion_summary["records"] / len(df) * 100
).round(2)

display(exclusion_summary)

count   855,028.00
mean        692.05
std       1,809.54
min     -41,318.00
1%            0.00
5%          225.00
25%         356.00
50%         485.00
75%         781.00
95%       1,700.00
99%       3,084.00
max     623,136.00
Name: response_time_seconds, dtype: float64

,exclusion_reason,records,percentage
0,kept,849637,77.56
1,missing_on_scene_dttm,240401,21.95
2,extreme_response_time_over_60_min,5340,0.49
3,negative_response_time,51,0.00


Important observations already:

- ~22% missing on_scene_dttm
- only 51 negative durations
- 60+ minute calls are rare (~0.5%)
- median response time ≈ 8 minutes
- strong right skew in the target distribution

## 7. Select Dispatch-Time Features Only

The client explicitly restricted usable features to dispatch-time information only.

In [10]:
# =========================
# 7. Select dispatch-time features only
# =========================

model_features = [
    "call_type",
    "call_type_group",
    "original_priority",
    "unit_id",
    "unit_type",
    "station_area",
    "battalion",
    "neighborhoods_analysis_boundaries",
    "zipcode_of_incident",
    "received_dttm",
]

target_column = "response_time_seconds"

df_model = df.loc[
    df["exclusion_reason"] == "kept",
    model_features + [target_column]
].copy()

print("Modelling dataframe shape:", df_model.shape)

display(df_model.head())

Modelling dataframe shape: (849637, 11)


,call_type,call_type_group,original_priority,unit_id,unit_type,station_area,battalion,neighborhoods_analysis_boundaries,zipcode_of_incident,received_dttm,response_time_seconds
0,Gas Leak (Natural and LP Gases),Alarm,3,T03,TRUCK,3.00,B01,Tenderloin,"94,102.00",2023-01-01 01:56:38,349.00
1,Medical Incident,Potentially Life-Threatening,3,52,MEDIC,17.00,B10,Bayview Hunters Point,"94,124.00",2023-01-01 02:10:43,902.00
2,Alarms,Alarm,3,B05,CHIEF,21.00,B05,Lone Mountain/USF,"94,118.00",2023-01-01 11:33:00,290.00
6,Medical Incident,Potentially Life-Threatening,3,62,MEDIC,7.00,B02,Mission,"94,110.00",2023-01-01 20:09:14,435.00
7,Medical Incident,Potentially Life-Threatening,3,55,MEDIC,13.00,B03,Financial District/South Beach,"94,105.00",2023-01-01 16:17:13,702.00


## 8. Engineer Time/Location Features

In [11]:
# =========================
# 8. Engineer time/location features
# =========================

# -------------------------
# Time features
# -------------------------

df_model["hour"] = df_model["received_dttm"].dt.hour
df_model["day_of_week"] = df_model["received_dttm"].dt.dayofweek
df_model["month"] = df_model["received_dttm"].dt.month
df_model["is_weekend"] = (
    df_model["day_of_week"] >= 5
).astype(int)

# -------------------------
# Rush hour feature
# -------------------------

df_model["is_rush_hour"] = (
    (
        (df_model["hour"].between(7, 9))
        |
        (df_model["hour"].between(16, 18))
    )
).astype(int)

# -------------------------
# Night shift feature
# -------------------------

df_model["is_night"] = (
    (
        (df_model["hour"] >= 22)
        |
        (df_model["hour"] <= 5)
    )
).astype(int)

# -------------------------
# Cyclical encoding
# -------------------------

df_model["hour_sin"] = np.sin(
    2 * np.pi * df_model["hour"] / 24
)

df_model["hour_cos"] = np.cos(
    2 * np.pi * df_model["hour"] / 24
)

df_model["dow_sin"] = np.sin(
    2 * np.pi * df_model["day_of_week"] / 7
)

df_model["dow_cos"] = np.cos(
    2 * np.pi * df_model["day_of_week"] / 7
)

# -------------------------
# Clean zipcode
# -------------------------

df_model["zipcode_of_incident"] = (
    df_model["zipcode_of_incident"]
    .fillna(0)
    .astype(int)
    .astype(str)
)

# -------------------------
# Convert station area
# -------------------------

df_model["station_area"] = (
    df_model["station_area"]
    .fillna(-1)
    .astype(int)
    .astype(str)
)

print("Feature engineering complete.")

display(df_model.head())

Feature engineering complete.


,call_type,call_type_group,original_priority,unit_id,unit_type,station_area,battalion,neighborhoods_analysis_boundaries,zipcode_of_incident,received_dttm,response_time_seconds,hour,day_of_week,month,is_weekend,is_rush_hour,is_night,hour_sin,hour_cos,dow_sin,dow_cos
0,Gas Leak (Natural and LP Gases),Alarm,3,T03,TRUCK,3,B01,Tenderloin,94102,2023-01-01 01:56:38,349.00,1,6,1,1,0,1,0.26,0.97,-0.78,0.62
1,Medical Incident,Potentially Life-Threatening,3,52,MEDIC,17,B10,Bayview Hunters Point,94124,2023-01-01 02:10:43,902.00,2,6,1,1,0,1,0.50,0.87,-0.78,0.62
2,Alarms,Alarm,3,B05,CHIEF,21,B05,Lone Mountain/USF,94118,2023-01-01 11:33:00,290.00,11,6,1,1,0,0,0.26,-0.97,-0.78,0.62
6,Medical Incident,Potentially Life-Threatening,3,62,MEDIC,7,B02,Mission,94110,2023-01-01 20:09:14,435.00,20,6,1,1,0,0,-0.87,0.50,-0.78,0.62
7,Medical Incident,Potentially Life-Threatening,3,55,MEDIC,13,B03,Financial District/South Beach,94105,2023-01-01 16:17:13,702.00,16,6,1,1,1,0,-0.87,-0.50,-0.78,0.62


The engineered features already look strong and production-appropriate:

- temporal operational patterns
- cyclical encoding
- dispatch-only inputs
- cleaned categorical/location features

## 9. Build Final Modelling Dataframe

Before splitting, we should:

- remove helper columns not needed
- ensure no missing values in modelling features
- inspect remaining categorical cardinality

In [12]:
# =========================
# 9. Build final modelling dataframe
# =========================

df_model_final = df_model.copy()

print("Final modelling dataframe shape:")
print(df_model_final.shape)

print("\nMissing values:")
display(df_model_final.isna().sum().sort_values(ascending=False).head(20))

Final modelling dataframe shape:
(849637, 21)

Missing values:


call_type_group                      16801
original_priority                     5330
neighborhoods_analysis_boundaries      153
unit_id                                  0
call_type                                0
unit_type                                0
station_area                             0
battalion                                0
zipcode_of_incident                      0
received_dttm                            0
response_time_seconds                    0
hour                                     0
day_of_week                              0
month                                    0
is_weekend                               0
is_rush_hour                             0
is_night                                 0
hour_sin                                 0
hour_cos                                 0
dow_sin                                  0
dtype: int64

In [13]:
# =========================
# Inspect categorical cardinality
# =========================

categorical_cols = [
    "call_type",
    "call_type_group",
    "original_priority",
    "unit_id",
    "unit_type",
    "station_area",
    "battalion",
    "neighborhoods_analysis_boundaries",
    "zipcode_of_incident",
]

cardinality = pd.DataFrame({
    "column": categorical_cols,
    "unique_values": [
        df_model_final[col].nunique()
        for col in categorical_cols
    ]
})

display(cardinality.sort_values(
    by="unique_values",
    ascending=False
))

,column,unique_values
3,unit_id,463
5,station_area,46
7,neighborhoods_analysis_boundaries,41
0,call_type,31
8,zipcode_of_incident,28
6,battalion,15
4,unit_type,12
2,original_priority,9
1,call_type_group,4


This is important because:

- XGBoost handles moderate cardinality well
- very high cardinality may require special encoding later

Before Step 10 we should properly handle the remaining missing values. Right now the modelling dataframe is not fully production-ready yet.

The good news:

- missingness is very small
- cardinality is completely manageable
- no dangerous high-cardinality explosion

Especially:

- 463 unique unit_id values is perfectly acceptable for tree models
- 41 neighborhoods is ideal for fairness analysis
- 31 call types is also manageable

This is actually a very healthy dataset for XGBoost/Random Forest.

---

What we should do now

We should NOT drop these rows.

Instead:

- preserve operational information
- create explicit "UNKNOWN" categories

This is better because:

- production APIs may receive incomplete data
- dropping rows may bias fairness analysis
- preserving missingness patterns is more realistic

This is also easier to justify in Report 2.

In [14]:
# =========================
# Handle remaining missing values
# =========================

df_model_final["call_type_group"] = (
    df_model_final["call_type_group"]
    .fillna("UNKNOWN")
)

df_model_final["original_priority"] = (
    df_model_final["original_priority"]
    .fillna("UNKNOWN")
)

df_model_final["neighborhoods_analysis_boundaries"] = (
    df_model_final["neighborhoods_analysis_boundaries"]
    .fillna("UNKNOWN")
)

print("Remaining missing values:")

display(
    df_model_final.isna().sum().sort_values(
        ascending=False
    ).head(20)
)

Remaining missing values:


call_type                            0
call_type_group                      0
original_priority                    0
unit_id                              0
unit_type                            0
station_area                         0
battalion                            0
neighborhoods_analysis_boundaries    0
zipcode_of_incident                  0
received_dttm                        0
response_time_seconds                0
hour                                 0
day_of_week                          0
month                                0
is_weekend                           0
is_rush_hour                         0
is_night                             0
hour_sin                             0
hour_cos                             0
dow_sin                              0
dtype: int64

## 10. Create Temporal Train/Validation/Test Split

This is CRITICAL.

Do NOT random split this dataset.

The client environment is time-dependent operational data.

We should simulate future predictions.

In [15]:
# =========================
# 10. Create temporal train/validation/test split
# =========================

# Sort chronologically first
df_model_final = df_model_final.sort_values("received_dttm").reset_index(drop=True)

print("Date range:")
print("Min received_dttm:", df_model_final["received_dttm"].min())
print("Max received_dttm:", df_model_final["received_dttm"].max())

Date range:
Min received_dttm: 2023-01-01 00:00:28
Max received_dttm: 2025-12-31 23:56:52


In [16]:
# =========================
# Define split boundaries
# =========================

train_end = pd.Timestamp("2024-06-30 23:59:59")
validation_end = pd.Timestamp("2024-12-31 23:59:59")

train_mask = df_model_final["received_dttm"] <= train_end

validation_mask = (
    (df_model_final["received_dttm"] > train_end)
    &
    (df_model_final["received_dttm"] <= validation_end)
)

test_mask = df_model_final["received_dttm"] > validation_end

In [17]:
# =========================
# Create split datasets
# =========================

train_df = df_model_final.loc[train_mask].copy()
validation_df = df_model_final.loc[validation_mask].copy()
test_df = df_model_final.loc[test_mask].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nSplit percentages:")
print("Train:", round(len(train_df) / len(df_model_final) * 100, 2), "%")
print("Validation:", round(len(validation_df) / len(df_model_final) * 100, 2), "%")
print("Test:", round(len(test_df) / len(df_model_final) * 100, 2), "%")

Train shape: (419905, 21)
Validation shape: (143813, 21)
Test shape: (285919, 21)

Split percentages:
Train: 49.42 %
Validation: 16.93 %
Test: 33.65 %


In [18]:
# =========================
# Verify temporal separation
# =========================

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "min_received_dttm": [
        train_df["received_dttm"].min(),
        validation_df["received_dttm"].min(),
        test_df["received_dttm"].min(),
    ],
    "max_received_dttm": [
        train_df["received_dttm"].max(),
        validation_df["received_dttm"].max(),
        test_df["received_dttm"].max(),
    ],
})

split_summary["percentage"] = (
    split_summary["rows"] / len(df_model_final) * 100
).round(2)

display(split_summary)

,split,rows,min_received_dttm,max_received_dttm,percentage
0,train,419905,2023-01-01 00:00:28,2024-06-30 23:48:59,49.42
1,validation,143813,2024-07-01 00:00:46,2024-12-31 23:56:18,16.93
2,test,285919,2025-01-01 00:07:27,2025-12-31 23:56:52,33.65


In [19]:
# =========================
# Sanity check target distribution by split
# =========================

target_by_split = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "target_mean_seconds": [
        train_df["response_time_seconds"].mean(),
        validation_df["response_time_seconds"].mean(),
        test_df["response_time_seconds"].mean(),
    ],
    "target_median_seconds": [
        train_df["response_time_seconds"].median(),
        validation_df["response_time_seconds"].median(),
        test_df["response_time_seconds"].median(),
    ],
    "target_p95_seconds": [
        train_df["response_time_seconds"].quantile(0.95),
        validation_df["response_time_seconds"].quantile(0.95),
        test_df["response_time_seconds"].quantile(0.95),
    ],
})

display(target_by_split.round(2))

,split,target_mean_seconds,target_median_seconds,target_p95_seconds
0,train,643.19,479.00,"1,614.00"
1,validation,647.67,486.00,"1,621.00"
2,test,653.20,489.00,"1,652.00"


The target distribution is stable across train, validation, and test:

- Train median:      479 sec
- Validation median: 486 sec
- Test median:       489 sec

So we can justify that the temporal split simulates future prediction while keeping comparable response-time patterns.

## 11. Save Processed Datasets

In [20]:
# =========================
# 11. Save processed datasets
# =========================

train_path = os.path.join(PROCESSED_DATA_DIR, "train_model_data.csv")
validation_path = os.path.join(PROCESSED_DATA_DIR, "validation_model_data.csv")
test_path = os.path.join(PROCESSED_DATA_DIR, "test_model_data.csv")
full_path = os.path.join(PROCESSED_DATA_DIR, "full_model_data.csv")

train_df.to_csv(train_path, index=False)
validation_df.to_csv(validation_path, index=False)
test_df.to_csv(test_path, index=False)
df_model_final.to_csv(full_path, index=False)

print("Saved files:")
print(train_path)
print(validation_path)
print(test_path)
print(full_path)

Saved files:
c:\Users\tozeq\Desktop\LDSSA - Capstone Project\Data\processed\train_model_data.csv
c:\Users\tozeq\Desktop\LDSSA - Capstone Project\Data\processed\validation_model_data.csv
c:\Users\tozeq\Desktop\LDSSA - Capstone Project\Data\processed\test_model_data.csv
c:\Users\tozeq\Desktop\LDSSA - Capstone Project\Data\processed\full_model_data.csv


## 12. Define X and y

In [21]:
# =========================
# 12. Define X and y
# =========================

target_column = "response_time_seconds"

drop_columns = [
    target_column,
    "received_dttm",
]

X_train = train_df.drop(columns=drop_columns)
y_train = train_df[target_column]

X_validation = validation_df.drop(columns=drop_columns)
y_validation = validation_df[target_column]

X_test = test_df.drop(columns=drop_columns)
y_test = test_df[target_column]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (419905, 19)
y_train: (419905,)
X_validation: (143813, 19)
y_validation: (143813,)
X_test: (285919, 19)
y_test: (285919,)


## 13. Build preprocessing pipeline

In [22]:
# =========================
# Define feature groups
# =========================

categorical_features = [
    "call_type",
    "call_type_group",
    "original_priority",
    "unit_id",
    "unit_type",
    "station_area",
    "battalion",
    "neighborhoods_analysis_boundaries",
    "zipcode_of_incident",
]

numerical_features = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_rush_hour",
    "is_night",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
]

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))

Categorical features: 9
Numerical features: 10


In [23]:
# =========================
# Preprocessing pipeline
# =========================

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
        (
            "numerical",
            numerical_transformer,
            numerical_features,
        ),
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


## 14. Baseline Linear Regression

In [24]:
# =========================
# 14. Baseline Linear Regression
# =========================

linear_regression_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

print("Training Linear Regression baseline...")

linear_regression_pipeline.fit(X_train, y_train)

print("Training complete.")

Training Linear Regression baseline...
Training complete.


In [25]:
# =========================
# Validation predictions
# =========================

validation_predictions_lr = (
    linear_regression_pipeline.predict(X_validation)
)

In [28]:
# =========================
# Baseline evaluation
# =========================

mae_lr = mean_absolute_error(
    y_validation,
    validation_predictions_lr,
)

mse_lr = mean_squared_error(
    y_validation,
    validation_predictions_lr,
)

rmse_lr = np.sqrt(mse_lr)

r2_lr = r2_score(
    y_validation,
    validation_predictions_lr,
)

baseline_results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "MAE": [round(mae_lr, 2)],
    "RMSE": [round(rmse_lr, 2)],
    "R2": [round(r2_lr, 4)],
})

display(baseline_results)

,Model,MAE,RMSE,R2
0,Linear Regression,237.97,389.54,0.35


Excellent baseline result.

For a first Linear Regression model on noisy operational emergency-response data:

| Metric | Result   |
| ------ | -------- |
| MAE    | ~238 sec |
| RMSE   | ~390 sec |
| R²     | 0.35     |

this is actually quite reasonable.

Interpretation for the future report:

> “The baseline linear model explained approximately 35% of the variance in response times, suggesting that operational response behavior contains substantial non-linear effects and complex interactions not fully captured by a linear formulation.”

Also:

```text
MAE ≈ 238 sec ≈ 4 minutes
```

So on average the baseline prediction misses by about 4 minutes.

That gives us a solid benchmark to beat with:

* Random Forest
* XGBoost

Now let’s move to the first non-linear model.

---



## 15. Random Forest Regressor

In [29]:
from sklearn.ensemble import RandomForestRegressor

In [31]:
# =========================
# 15. Random Forest baseline
# =========================

# random_forest_pipeline = Pipeline(
#     steps=[
#         ("preprocessor", preprocessor),
#         (
#             "model",
#             RandomForestRegressor(
#                 n_estimators=100,
#                 max_depth=20,
#                 min_samples_split=10,
#                 min_samples_leaf=5,
#                 random_state=42,
#                 n_jobs=-1,
#             ),
#         ),
#     ]
# )

# print("Training Random Forest...")

# random_forest_pipeline.fit(X_train, y_train)

# print("Training complete.")

# =========================
# 15. Random Forest baseline (lighter version)
# =========================

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=30,
                max_depth=12,
                min_samples_split=20,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

print("Training Random Forest...")

random_forest_pipeline.fit(X_train, y_train)

print("Training complete.")

Training Random Forest...
Training complete.


In [32]:
# =========================
# Random Forest predictions
# =========================

validation_predictions_rf = (
    random_forest_pipeline.predict(X_validation)
)

In [33]:
# =========================
# Random Forest evaluation
# =========================

mae_rf = mean_absolute_error(
    y_validation,
    validation_predictions_rf,
)

mse_rf = mean_squared_error(
    y_validation,
    validation_predictions_rf,
)

rmse_rf = np.sqrt(mse_rf)

r2_rf = r2_score(
    y_validation,
    validation_predictions_rf,
)

rf_results = pd.DataFrame({
    "Model": ["Random Forest"],
    "MAE": [round(mae_rf, 2)],
    "RMSE": [round(rmse_rf, 2)],
    "R2": [round(r2_rf, 4)],
})

display(rf_results)

,Model,MAE,RMSE,R2
0,Random Forest,230.64,377.91,0.39


In [34]:
# =========================
# Model comparison
# =========================

comparison_results = pd.concat(
    [
        baseline_results,
        rf_results,
    ],
    ignore_index=True,
)

display(comparison_results)

,Model,MAE,RMSE,R2
0,Linear Regression,237.97,389.54,0.35
1,Random Forest,230.64,377.91,0.39


Excellent — and actually very informative.

The Random Forest improved performance, but only moderately:

| Model             | MAE    | RMSE   | R²   |
| ----------------- | ------ | ------ | ---- |
| Linear Regression | 237.97 | 389.54 | 0.35 |
| Random Forest     | 230.64 | 377.91 | 0.39 |

This tells us something important about the problem:

```text
The dataset contains signal, but also substantial operational noise.
```

This is exactly the kind of realistic conclusion the capstone expects. 

---

### Important interpretation

The Random Forest:

* captures some non-linearities
* improves variance explanation
* reduces large errors slightly

BUT:

* gains are not dramatic
* operational unpredictability remains high

This is actually a strong report narrative because:

* you are not overselling the model
* you are showing critical thinking
* you can discuss limits realistically

---




## 16. XGBoost Regressor

In [35]:
from xgboost import XGBRegressor

In [36]:
# =========================
# 16. XGBoost baseline
# =========================

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

print("Training XGBoost...")

xgb_pipeline.fit(X_train, y_train)

print("Training complete.")

Training XGBoost...
Training complete.


In [37]:
# =========================
# XGBoost predictions
# =========================

validation_predictions_xgb = (
    xgb_pipeline.predict(X_validation)
)

In [38]:
# =========================
# XGBoost evaluation
# =========================

mae_xgb = mean_absolute_error(
    y_validation,
    validation_predictions_xgb,
)

mse_xgb = mean_squared_error(
    y_validation,
    validation_predictions_xgb,
)

rmse_xgb = np.sqrt(mse_xgb)

r2_xgb = r2_score(
    y_validation,
    validation_predictions_xgb,
)

xgb_results = pd.DataFrame({
    "Model": ["XGBoost"],
    "MAE": [round(mae_xgb, 2)],
    "RMSE": [round(rmse_xgb, 2)],
    "R2": [round(r2_xgb, 4)],
})

display(xgb_results)

,Model,MAE,RMSE,R2
0,XGBoost,219.77,360.52,0.44


In [39]:
# =========================
# Full model comparison
# =========================

comparison_results = pd.concat(
    [
        baseline_results,
        rf_results,
        xgb_results,
    ],
    ignore_index=True,
)

display(comparison_results)

,Model,MAE,RMSE,R2
0,Linear Regression,237.97,389.54,0.35
1,Random Forest,230.64,377.91,0.39
2,XGBoost,219.77,360.52,0.44




Your results show a clear and technically coherent progression:

| Model             | MAE    | RMSE   | R²   |
| ----------------- | ------ | ------ | ---- |
| Linear Regression | 237.97 | 389.54 | 0.35 |
| Random Forest     | 230.64 | 377.91 | 0.39 |
| XGBoost           | 219.77 | 360.52 | 0.44 |

---

### Interpretation

The XGBoost model:

* achieved the best overall predictive performance
* reduced average prediction error by ~18 seconds vs Linear Regression
* improved explained variance from 35% → 44%

This is very realistic and credible for emergency-response operational data.

Most importantly:

* improvements are meaningful
* but not suspiciously perfect

That’s exactly what graders expect in a real-world capstone.

---

### Important insight for the report

You now have strong evidence supporting:

```text id="t4pq3n"
Emergency response times are influenced by complex non-linear operational interactions.
```

Which explains why:

* tree boosting outperformed linear methods
* purely linear relationships were insufficient

---




## 17. Fairness & subgroup performance analysis

Before choosing the final model officially, we should now do Fairness & subgroup performance analysis

This is VERY important because the client explicitly requested it.

### Goal

We want to answer:

Does model performance vary across:
- neighborhoods
- battalions
- call type groups

If yes:

- how much?
- where?
- is there operational inequity?

### 17.1 Build evaluation dataframe

In [40]:
# =========================
# Fairness analysis dataframe
# =========================

evaluation_df = validation_df.copy()

evaluation_df["prediction_xgb"] = validation_predictions_xgb

evaluation_df["absolute_error"] = np.abs(
    evaluation_df["response_time_seconds"]
    -
    evaluation_df["prediction_xgb"]
)

display(evaluation_df.head())

,call_type,call_type_group,original_priority,unit_id,unit_type,station_area,battalion,neighborhoods_analysis_boundaries,zipcode_of_incident,received_dttm,response_time_seconds,hour,day_of_week,month,is_weekend,is_rush_hour,is_night,hour_sin,hour_cos,dow_sin,dow_cos,prediction_xgb,absolute_error
419905,Alarms,Alarm,3,B01,CHIEF,3,B04,Tenderloin,94109,2024-07-01 00:00:46,527.00,0,0,7,0,0,1,0.00,1.00,0.00,1.00,411.12,115.88
419906,Alarms,Alarm,3,T03,TRUCK,3,B04,Tenderloin,94109,2024-07-01 00:00:46,336.00,0,0,7,0,0,1,0.00,1.00,0.00,1.00,370.53,34.53
419907,Alarms,Alarm,3,E03,ENGINE,3,B04,Tenderloin,94109,2024-07-01 00:00:46,340.00,0,0,7,0,0,1,0.00,1.00,0.00,1.00,327.17,12.83
419908,Other,UNKNOWN,A,SCRT9,CP,22,B08,Sunset/Parkside,94122,2024-07-01 00:01:05,"1,191.00",0,0,7,0,0,1,0.00,1.00,0.00,1.00,"1,483.90",292.90
419909,Medical Incident,Potentially Life-Threatening,A,AM122,PRIVATE,11,B06,Bernal Heights,94110,2024-07-01 00:02:02,305.00,0,0,7,0,0,1,0.00,1.00,0.00,1.00,509.01,204.01


### 17.2 — Neighborhood performance

In [41]:
# =========================
# MAE by neighborhood
# =========================

neighborhood_performance = (
    evaluation_df
    .groupby("neighborhoods_analysis_boundaries")
    .agg(
        records=("response_time_seconds", "size"),
        actual_mean_response=("response_time_seconds", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)

# Keep only sufficiently large groups
neighborhood_performance = (
    neighborhood_performance[
        neighborhood_performance["records"] >= 500
    ]
)

neighborhood_performance = (
    neighborhood_performance
    .sort_values("mae", ascending=False)
)

display(neighborhood_performance.head(15))

,neighborhoods_analysis_boundaries,records,actual_mean_response,mae
29,Presidio,991,814.92,266.51
36,Treasure Island,724,758.50,266.24
4,Excelsior,2567,754.96,241.23
21,Noe Valley,1724,598.92,236.87
5,Financial District/South Beach,8879,701.34,234.54
0,Bayview Hunters Point,7475,721.33,233.05
20,Nob Hill,5418,564.82,232.52
19,Mission Bay,2900,682.16,232.50
11,Inner Sunset,1991,644.73,231.79
33,South of Market,16323,661.44,231.19


### 17.3 Battalion performance

In [42]:
# =========================
# MAE by battalion
# =========================

battalion_performance = (
    evaluation_df
    .groupby("battalion")
    .agg(
        records=("response_time_seconds", "size"),
        actual_mean_response=("response_time_seconds", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
    .sort_values("mae", ascending=False)
)

display(battalion_performance)

,battalion,records,actual_mean_response,mae
0,AMB,1,0.00,605.39
11,B99,16,254.81,508.37
12,XXX,2,4.00,375.41
10,B10,11597,691.20,235.97
3,B03,26227,684.55,231.08
9,B09,9560,720.00,223.94
8,B08,10956,707.34,223.06
7,B07,7664,661.70,221.25
6,B06,9961,629.90,220.51
1,B01,14098,628.15,212.43


### 17.4 Call type group performance

In [43]:
# =========================
# MAE by call type group
# =========================

call_group_performance = (
    evaluation_df
    .groupby("call_type_group")
    .agg(
        records=("response_time_seconds", "size"),
        actual_mean_response=("response_time_seconds", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
    .sort_values("mae", ascending=False)
)

display(call_group_performance)

,call_type_group,records,actual_mean_response,mae
4,UNKNOWN,3199,733.86,563.61
2,Non Life-threatening,31677,"1,016.34",358.45
1,Fire,5976,559.56,209.93
3,Potentially Life-Threatening,70330,569.85,188.49
0,Alarm,32631,465.20,120.66



---

### Key findings you should already note

#### 1. Performance varies meaningfully by operational context

Example:

| Call Group                   | MAE     |
| ---------------------------- | ------- |
| Alarm                        | 121 sec |
| Potentially Life-Threatening | 188 sec |
| Non Life-threatening         | 358 sec |

This is extremely important.

Interpretation:

```text id="hz9bl5"
More operationally complex incidents are substantially harder to predict.
```

That is realistic and analytically valuable.

---

#### 2. Neighborhood disparities remain

Neighborhood MAE ranges:

* ~225 sec
* up to ~266 sec

This suggests:

* operational heterogeneity
* geographic variability
* possible resource imbalance
* differing traffic/access conditions

Exactly aligned with the client’s concerns. 

---

#### 3. Rare categories behave poorly

The `"UNKNOWN"` call group:

* very high MAE
* likely due to sparse data
* operational ambiguity

This is excellent material for:

* Known issues and risks
* deployment limitations

---

#### 4. Small battalions distort metrics

These:

```text id="5f2oq4"
AMB
B99
XXX
```

have tiny sample sizes.

Important:
DO NOT over-interpret them.

In the report you should explicitly mention:

* low-support groups create unstable estimates

This demonstrates statistical maturity.

---

### VERY IMPORTANT CONCLUSION

Your model is NOT uniformly accurate.

That is:

* expected
* realistic
* analytically valuable

And directly answers the client’s fairness concerns.

---

### Next Step — Feature Importance

Now we need explainability.

This is crucial because:

* XGBoost is your likely final model
* stakeholders need interpretability
* Report 2 expects technical transparency 

---




## 18. Extract feature importance

In [44]:
# =========================
# 18. Feature importance
# =========================

# Get trained XGBoost model
xgb_model = xgb_pipeline.named_steps["model"]

# Get transformed feature names
feature_names = (
    xgb_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Create feature importance dataframe
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_model.feature_importances_,
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
)

display(feature_importance.head(20))

,feature,importance
32,categorical__call_type_group_Non Life-threatening,0.08
446,categorical__unit_type_INVESTIGATION,0.05
35,categorical__original_priority_1,0.05
36,categorical__original_priority_2,0.04
445,categorical__unit_type_ENGINE,0.04
37,categorical__original_priority_3,0.03
444,categorical__unit_type_CP,0.03
34,categorical__call_type_group_UNKNOWN,0.02
452,categorical__unit_type_TRUCK,0.02
174,categorical__unit_id_AR1,0.02


In [46]:
# =========================
# Aggregate feature importance
# =========================


def map_feature_group(feature_name):

    if "call_type_group" in feature_name:
        return "call_type_group"

    elif "call_type_" in feature_name:
        return "call_type"

    elif "original_priority" in feature_name:
        return "original_priority"

    elif "unit_id" in feature_name:
        return "unit_id"

    elif "unit_type" in feature_name:
        return "unit_type"

    elif "station_area" in feature_name:
        return "station_area"

    elif "battalion" in feature_name:
        return "battalion"

    elif "neighborhoods_analysis_boundaries" in feature_name:
        return "neighborhood"

    elif "zipcode_of_incident" in feature_name:
        return "zipcode"

    elif "hour" in feature_name:
        return "hour"

    elif "day_of_week" in feature_name:
        return "day_of_week"

    elif "month" in feature_name:
        return "month"

    else:
        return "other"


feature_importance["feature_group"] = (
    feature_importance["feature"]
    .apply(map_feature_group)
)

aggregated_importance = (
    feature_importance
    .groupby("feature_group")["importance"]
    .sum()
    .reset_index()
    .sort_values("importance", ascending=False)
)

display(aggregated_importance)

,feature_group,importance
10,unit_id,0.32
11,unit_type,0.17
7,original_priority,0.15
2,call_type_group,0.13
1,call_type,0.05
9,station_area,0.04
6,neighborhood,0.04
12,zipcode,0.04
0,battalion,0.03
4,hour,0.00




These results are honestly excellent for Report 2 because they tell a coherent operational story.

### Key operational insight

The strongest predictors are:

| Feature Group     | Importance |
| ----------------- | ---------- |
| unit_id           | 32%        |
| unit_type         | 17%        |
| original_priority | 15%        |
| call_type_group   | 13%        |

This strongly suggests:

```text id="v4d37v"
Response time is driven primarily by operational deployment characteristics rather than temporal seasonality.
```

That is a sophisticated and realistic finding.

---

### VERY IMPORTANT observation

Time features being near-zero is actually valuable.

It means:

* the system is relatively operationally stable across time
* deployment/resource configuration matters more
* dispatch logic dominates temporal effects

This is a much stronger conclusion than:

> “hour matters a bit”

---

### Another important operational implication

The dominance of:

* unit_id
* unit_type

also suggests:

```text id="1t6q63"
Certain units consistently operate under different response conditions.
```

Possible reasons:

* geographic positioning
* specialized equipment
* dispatch protocol
* staffing levels
* operational role

This connects beautifully to:

* fairness analysis
* deployment risks
* future optimization recommendations

---

#### At this point your modelling phase is already very strong

You now have:

* proper temporal split
* realistic target engineering
* baseline comparison
* non-linear model improvement
* fairness analysis
* explainability
* operational insights


---




## 19. Export production artifacts

In [47]:
# =========================
# 19. Create models directory
# =========================

MODELS_DIR = os.path.join(PROJECT_DIR, "models")

os.makedirs(MODELS_DIR, exist_ok=True)

print("Models directory:", MODELS_DIR)

Models directory: c:\Users\tozeq\Desktop\LDSSA - Capstone Project\models


In [48]:
# =========================
# Save final pipeline
# =========================

import joblib

pipeline_path = os.path.join(
    MODELS_DIR,
    "xgb_response_time_pipeline.pkl"
)

joblib.dump(
    xgb_pipeline,
    pipeline_path,
)

print("Pipeline saved to:")
print(pipeline_path)

Pipeline saved to:
c:\Users\tozeq\Desktop\LDSSA - Capstone Project\models\xgb_response_time_pipeline.pkl


In [49]:
# =========================
# Save feature metadata
# =========================

feature_metadata = {
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "target_column": target_column,
}

metadata_path = os.path.join(
    MODELS_DIR,
    "feature_metadata.pkl"
)

joblib.dump(
    feature_metadata,
    metadata_path,
)

print("Feature metadata saved.")

Feature metadata saved.


In [50]:
# =========================
# Reload pipeline test
# =========================

loaded_pipeline = joblib.load(pipeline_path)

sample_predictions = loaded_pipeline.predict(
    X_validation.head(5)
)

print("Reload successful.")

print(sample_predictions)

Reload successful.
[ 411.1179   370.52618  327.1662  1483.8997   509.01068]
